# LAK-4 — Orphan files & GC

**Break → Detect → Fix → Prove.** Failed/aborted writes, cancelled compactions, or direct path
writes leave **data files on disk that no snapshot references** — *orphans*. They consume storage
forever, and **snapshot expiry does NOT reclaim them** (they were never referenced).

**Pre-requisite:** `make up` → Spark UI at http://localhost:4040. **Laptop-safe:** tiny data under
`.tmp/`; `make clean` recovers. The Iceberg warehouse is `./.tmp/local_iceberg_warehouse`.

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import os, glob, time, shutil
import common

T = "iceberg_catalog.default.lak4_t"
# Spark (server CWD=/app) writes here; the client (kernel CWD = this notebook's dir) must
# anchor the SAME files to the repo root — derive it from the imported `common` package.
SERVER_DATA = ".tmp/local_iceberg_warehouse/default/lak4_t/data"        # for Spark writes
_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(common.__file__)))
TBL_DIR = os.path.join(_ROOT, ".tmp/local_iceberg_warehouse/default/lak4_t")
DATA = os.path.join(TBL_DIR, "data")                                    # for client-side glob/os
def disk_files():
    return sorted(glob.glob(DATA + "/*.parquet"))
spark

## Step 0 — a clean table

On a healthy table, **on-disk data files == metadata `data_files`** (every file is referenced).

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
shutil.rmtree(TBL_DIR, ignore_errors=True)  # clear orphans from any prior run
spark.sql(f"CREATE TABLE {T} (id BIGINT, v DOUBLE) USING iceberg")
spark.range(1000).withColumn("v", F.rand()).writeTo(T).append()

real_files = disk_files()                                  # the table's referenced data files
meta = table_health(spark, T, "clean")
print(f"on-disk parquet: {len(real_files)}   metadata data_files: {meta['data_files']}   (match = healthy)")

## 1. Break it — leave orphan files behind

A failed write / cancelled job / direct write to the table path drops parquet files that no
snapshot ever records. We simulate that by writing extra parquet **straight into the data dir**.

In [ ]:
spark.range(5000).withColumn("v", F.rand()).write.mode("append").parquet(SERVER_DATA)   # NOT a table write
orphans = [f for f in disk_files() if f not in real_files]
broken = table_health(spark, T, "after orphan write")
print(f"on-disk parquet: {len(disk_files())}   metadata data_files: {broken['data_files']}")
print(f"=> {len(orphans)} ORPHAN file(s): on disk, eating storage, referenced by NO snapshot")

## 2. Detect it

**On-disk file count > metadata `data_files`** is the signature — storage grows with no new rows.
And note: **`expire_snapshots` does NOT remove orphans** (it only drops files de-referenced by
*expired snapshots*; orphans were never referenced at all).

In [ ]:
spark.sql(f"CALL iceberg_catalog.system.expire_snapshots(table => 'default.lak4_t', retain_last => 1)")
print(f"on-disk parquet after expire_snapshots: {len(disk_files())}  (orphans still here — expiry can't reclaim them)")

## 3. Fix it — `remove_orphan_files` (mind the 24-hour safety guard)

`remove_orphan_files` deletes unreferenced files — but it **refuses any `older_than` within 24 h**
(modified files could be in-flight writes it would corrupt). So you can only reclaim files older
than that window.

> **Teaching shortcut (NEVER do this in prod):** we *backdate the orphans' mtimes* so they look old,
> then pass `older_than = 2 days ago` (which clears the guard). Real, fresh table files are protected
> two ways — they're **referenced** *and* **newer** than the threshold. In production you schedule
> this with a conservative age (e.g. 3–7 days) and never override the guard.

In [ ]:
old = time.time() - 3 * 86400                              # 3 days ago
for f in orphans:                                          # backdate ONLY the orphans
    os.utime(f, (old, old))

older_than = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d %H:%M:%S")   # clears the 24h guard
removed = spark.sql(
    f"CALL iceberg_catalog.system.remove_orphan_files("
    f"table => 'default.lak4_t', older_than => TIMESTAMP '{older_than}')"
).collect()
print(f"remove_orphan_files deleted {len(removed)} orphan file(s):")
for r in removed[:5]:
    print("  -", r[0].split('/')[-1])

## 4. Prove it

In [ ]:
fixed = table_health(spark, T, "after remove_orphan_files")
fresh_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {T}").first()["n"]   # re-reads from disk -> proves integrity
print(f"{'stage':<26}{'on-disk':>9}{'metadata':>10}{'orphans':>9}")
print(f"{'clean (start)':<26}{len(real_files):>9}{meta['data_files']:>10}{0:>9}")
print(f"{'after orphan write':<26}{len(real_files)+len(orphans):>9}{broken['data_files']:>10}{len(orphans):>9}")
print(f"{'after remove_orphan':<26}{len(disk_files()):>9}{fixed['data_files']:>10}{len(disk_files())-fixed['data_files']:>9}")
assert len(disk_files()) == fixed["data_files"], "on-disk should == metadata after GC"
assert fresh_rows == 1000, "real data must be intact"
print(f"\nPROVED: orphans reclaimed (on-disk == metadata), real data intact ({fresh_rows:,} rows).")

## Takeaways & "in real production…"

- **Orphans aren't reclaimed by snapshot expiry** — you need `remove_orphan_files`.
- Its **24 h guard** is a safety feature: it refuses to delete recently-modified files (possible
  in-flight writes). Schedule the procedure with a **conservative `older_than`** (days), never `now()`.
- Sources of orphans: failed/killed writes, cancelled compaction, direct writes to the table path.
- Pair with `expire_snapshots` (LAK-3) and compaction (LAK-2) as routine table maintenance; see `gc.*` table props.

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
shutil.rmtree(TBL_DIR, ignore_errors=True)
print("Dropped table + cleared its dir. (`make clean` clears all of .tmp.)")